# AMPS II · Week 03 — Lab
## Generalized plane strain, static condensation, one-way and fully coupled thermoelasticity

| | |
|---|---|
| **Lecture sections** | Part A (three 2D reductions, GPS, thermal stress, coupling number) · Part B (GPS code, static condensation) · Part C (thermal code, mixed space) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | solve the thick tube as a cross-section with one extra unknown $e_0$ for three end conditions · assemble the bordered matrix and compare it with static condensation · find out what the supports of a heated plate do · see a fully coupled transient run slower than the one-way model, by exactly $1+\delta$ |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The two model modules are written once
with `%%writefile` (§1) and imported by every `%%fenicsx` cell. Results pass through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model modules  *(lecture §B1–B3, §C1, §C4)*
`gps.py`: a mapped quarter annulus with curved cells (no Gmsh — the same mesh on every computer), rectangles, `Material` (numbers or per-region),
the 3D law with $\varepsilon_{zz}$ chosen by the reduction, `solve_2d` (plane strain, plane stress, GPS by static condensation), steady `heat`.
`coupled.py`: the transient $(\theta,\mathbf u)$ strip in one mixed space, the coupling number $\delta$ and the exact series.
Read them once; run both cells — they only write files.

In [ ]:
%%writefile /tmp/fnx/gps.py
"""2D reductions of a long prism with ONE 3D law. The cross-section is the (x, y) plane, z is the axis of the prism.
eps_zz = 0 (plane strain), eps_zz = e0 with int sigma_zz dA = N (generalized plane strain, GPS), sigma_zz = 0 (plane stress).
Thermal strain alpha*dT; steady heat conduction; meshes: a mapped quarter annulus with curved cells, rectangles."""
import json, os
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

INNER, OUTER, BOTTOM, LEFT, RIGHT, TOP = 1, 2, 3, 4, 5, 6

def annulus_mesh(ri, ro, nr, nt):
    """quarter annulus ri <= r <= ro, 0 <= theta <= pi/2: nr x nt cells in (r, theta), each split in two triangles,
    6-node (P2) geometry mapped from (r, theta) -> curved arcs, and the same nodes on every machine (no Gmsh).
    Facet tags: 1 r = ri, 2 r = ro, 3 y = 0, 4 x = 0."""
    if comm.rank == 0:
        nI, nJ = 2 * nr + 1, 2 * nt + 1
        R, TH = np.meshgrid(np.linspace(ri, ro, nI), np.linspace(0, np.pi / 2, nJ), indexing="ij")
        x = np.c_[(R * np.cos(TH)).ravel(), (R * np.sin(TH)).ravel()]
        x[np.abs(x) < 1e-14] = 0.0
        node = lambda p: p[0] * nJ + p[1]
        mid = lambda p, q: ((p[0] + q[0]) // 2, (p[1] + q[1]) // 2)
        cells = []
        for a in range(nr):
            for b in range(nt):
                i, j = 2 * a, 2 * b
                for p, q, s in (((i, j), (i + 2, j), (i + 2, j + 2)), ((i, j), (i + 2, j + 2), (i, j + 2))):
                    cells.append([node(p), node(q), node(s), node(mid(q, s)), node(mid(p, s)), node(mid(p, q))])  # basix order
        cells = np.array(cells, dtype=np.int64)
    else:
        x, cells = np.zeros((0, 2)), np.zeros((0, 6), dtype=np.int64)
    msh = mesh.create_mesh(comm, cells, ufl.Mesh(basix.ufl.element("Lagrange", "triangle", 2, shape=(2,))), x)
    rad = lambda X: np.hypot(X[0], X[1])
    return msh, _tags(msh, [(INNER, lambda X: np.isclose(rad(X), ri)), (OUTER, lambda X: np.isclose(rad(X), ro)),
                            (BOTTOM, lambda X: np.isclose(X[1], 0.0)), (LEFT, lambda X: np.isclose(X[0], 0.0))])

def rect_mesh(L, H, nx, ny, x0=0.0, y0=0.0):
    """rectangle [x0, x0+L] x [y0, y0+H], triangles. Facet tags: 3 bottom, 4 left, 5 right, 6 top"""
    msh = mesh.create_rectangle(comm, [[x0, y0], [x0 + L, y0 + H]], [nx, ny], cell_type=mesh.CellType.triangle)
    return msh, _tags(msh, [(BOTTOM, lambda X: np.isclose(X[1], y0)), (LEFT, lambda X: np.isclose(X[0], x0)),
                            (RIGHT, lambda X: np.isclose(X[0], x0 + L)), (TOP, lambda X: np.isclose(X[1], y0 + H))])

def _tags(msh, marks):
    fac, val = [], []
    for tag, f in marks:
        e = mesh.locate_entities_boundary(msh, 1, f); fac.append(e); val.append(np.full(len(e), tag, dtype=np.int32))
    fac, idx = np.unique(np.concatenate(fac), return_index=True)
    return mesh.meshtags(msh, 1, fac, np.concatenate(val)[idx])

def area(msh):
    return comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ufl.dx(domain=msh))), op=MPI.SUM)

def lame_constants(E, nu):
    return E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu))

class Material:
    """isotropic with thermal expansion alpha; beta = (3 lambda + 2 mu) alpha is the 3D thermal stress modulus.
    Material(E, nu, alpha) for one material; Material.regions(msh, parts) for piecewise-constant data (DG0 functions)."""
    def __init__(self, E=1.0, nu=0.3, alpha=0.0):
        self.E, self.nu, self.alpha = E, nu, alpha
        self.mu, self.lmbda = lame_constants(E, nu)
        self.beta = (3 * self.lmbda + 2 * self.mu) * alpha

    @staticmethod
    def regions(msh, parts):
        """parts = [(marker(x) -> bool, evaluated at cell midpoints, E, nu, alpha), ...]; later parts overwrite earlier ones"""
        Q = fem.functionspace(msh, ("DG", 0)); m = Material()
        n = msh.topology.index_map(2).size_local + msh.topology.index_map(2).num_ghosts
        mid = mesh.compute_midpoints(msh, 2, np.arange(n, dtype=np.int32)).T
        f = {k: fem.Function(Q) for k in ("E", "nu", "alpha", "mu", "lmbda", "beta")}
        for marker, E, nu, alpha in parts:
            dofs = Q.dofmap.list[np.flatnonzero(marker(mid))].ravel()
            mu, lm = lame_constants(E, nu)
            for k, val in (("E", E), ("nu", nu), ("alpha", alpha), ("mu", mu), ("lmbda", lm), ("beta", (3 * lm + 2 * mu) * alpha)):
                f[k].x.array[dofs] = val
        for k, fun in f.items():
            setattr(m, k, fun)
        return m

def eps3(u, ezz):
    """3D strain of the cross-section: in-plane part from u = (u_x, u_y), eps_zz = ezz, no z-shear"""
    g = (u[0].dx(1) + u[1].dx(0)) / 2
    return ufl.as_matrix([[u[0].dx(0), g, 0], [g, u[1].dx(1), 0], [0, 0, ezz]])

def sigma3(m, u, ezz, dT=0.0):
    """the 3D law  sigma = 2 mu eps + lambda tr(eps) I - beta dT I  (all three reductions use it)"""
    e = eps3(u, ezz)
    return 2 * m.mu * e + m.lmbda * ufl.tr(e) * ufl.Identity(3) - m.beta * dT * ufl.Identity(3)

def ezz_of(m, mode, u, dT, e0):
    """a reduction is a choice of eps_zz: plane strain 0; GPS the unknown number e0; plane stress the value that makes sigma_zz = 0"""
    if mode == "strain":
        return 0.0
    if mode == "gps":
        return e0
    return (m.beta * dT - m.lmbda * ufl.div(u)) / (m.lmbda + 2 * m.mu)

def solve_2d(msh, ft, mat, deg=2, mode="gps", N=0.0, tractions=(), body=None, dT=0.0, fixed=(), points=()):
    """mode 'strain' | 'stress' | 'gps'.  tractions [(tag, (tx, ty))] (numbers or UFL); body force (fx, fy) or None; dT temperature change (number, UFL or Function);
    fixed [(tag, comp)]: u_comp = 0 on those facets; points [(marker, comp)]: u_comp = 0 at the vertices where marker is True.
    GPS by STATIC CONDENSATION of e0: case 1 (loads on, e0 = 0) gives u0 and N0 = int sigma_zz; case 2 (loads off, e0 = 1) gives u1 and N1.
    Both cases have the SAME matrix: one LU factorisation, two solves.  N0 + e0 N1 = N gives e0, and u = u0 + e0 u1.
    Returns uh and info: e0, N0, N1, N (check), sigma (3x3 UFL), the matrix A and the two right-hand sides rhs (for checks)."""
    V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    ds = ufl.Measure("ds", domain=msh, subdomain_data=ft); dx = ufl.Measure("dx", domain=msh)
    lf, e0c = fem.Constant(msh, 1.0), fem.Constant(msh, 0.0)          # load factor and e0 of the current case
    zero = fem.Constant(msh, np.zeros(2))
    S2 = lambda S: ufl.as_matrix([[S[0, 0], S[0, 1]], [S[1, 0], S[1, 1]]])
    ezz_u = ezz_of(mat, mode, u, 0.0, 0.0) if mode == "stress" else 0.0     # the part of eps_zz that depends on u
    a = ufl.inner(S2(sigma3(mat, u, ezz_u)), ufl.sym(ufl.grad(v))) * dx
    L = ufl.dot(zero, v) * dx - ufl.inner(S2(sigma3(mat, zero, ezz_of(mat, mode, zero, lf * dT, e0c), lf * dT)), ufl.sym(ufl.grad(v))) * dx
    for tag, t in tractions:
        L += lf * ufl.dot(ufl.as_vector(t), v) * ds(tag)
    if body is not None:
        L += lf * ufl.dot(ufl.as_vector(body), v) * dx
    bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(c), 1, ft.find(tag)), V.sub(c)) for tag, c in fixed]
    for marker, c in points:
        bcs.append(fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(c), 0, mesh.locate_entities_boundary(msh, 0, marker)), V.sub(c)))
    a, L = fem.form(a), fem.form(L)
    A = assemble_matrix(a, bcs=bcs); A.assemble()
    ksp = PETSc.KSP().create(comm); ksp.setOperators(A); ksp.setType("preonly")
    ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    rhs = []
    def solve(lf_, e0_):
        lf.value, e0c.value = lf_, e0_
        b = assemble_vector(L); apply_lifting(b, [a], [bcs])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, bcs); rhs.append(b)
        w = fem.Function(V); ksp.solve(b, w.x.petsc_vec); w.x.scatter_forward()
        return w
    Nz = lambda w, e0_, lf_: comm.allreduce(fem.assemble_scalar(fem.form(
        sigma3(mat, w, ezz_of(mat, mode, w, lf_ * dT, e0_), lf_ * dT)[2, 2] * dx)), op=MPI.SUM)
    uh = solve(1.0, 0.0)
    info = dict(e0=0.0, A=A, rhs=rhs, reason=ksp.getConvergedReason(), dofs=V.dofmap.index_map.size_global * 2, V=V, ds=ds, dx=dx)
    if mode == "gps":
        u1 = solve(0.0, 1.0)
        N0, N1 = Nz(uh, 0.0, 1.0), Nz(u1, 1.0, 0.0)
        info.update(e0=(N - N0) / N1, N0=N0, N1=N1)
        uh.x.array[:] += info["e0"] * u1.x.array
    info["N"] = Nz(uh, info["e0"], 1.0)
    info["sigma"] = sigma3(mat, uh, ezz_of(mat, mode, uh, dT, info["e0"]), dT)
    return uh, info

def heat(msh, ft, k=1.0, Q=0.0, dirichlet=(), deg=1):
    """steady  -div(k grad T) = Q,  T = value on the tagged facets [(tag, value)], insulated elsewhere"""
    VT = fem.functionspace(msh, ("Lagrange", deg))
    T, s = ufl.TrialFunction(VT), ufl.TestFunction(VT)
    dx = ufl.Measure("dx", domain=msh)
    bcs = [fem.dirichletbc(fem.Constant(msh, float(val)), fem.locate_dofs_topological(VT, 1, ft.find(tag)), VT) for tag, val in dirichlet]
    pr = LinearProblem(k * ufl.inner(ufl.grad(T), ufl.grad(s)) * dx, Q * s * dx + fem.Constant(msh, 0.0) * s * dx, bcs=bcs,
                       petsc_options_prefix="heat_", petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    return pr.solve()

def von_mises(S):
    s = ufl.dev(S)
    return ufl.sqrt(1.5 * ufl.inner(s, s))

def dg_max(msh, expr, deg=1):
    """maximum of a scalar UFL expression over the domain, from its DG(deg) interpolant (call on all ranks)"""
    W = fem.functionspace(msh, ("DG", deg)); f = fem.Function(W)
    f.interpolate(fem.Expression(expr, W.element.interpolation_points))
    return comm.allreduce(float(f.x.array.max()) if f.x.array.size else -np.inf, op=MPI.MAX)

def l2(msh, expr):
    return np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(ufl.inner(expr, expr) * ufl.dx(domain=msh))), op=MPI.SUM))

In [ ]:
%%writefile /tmp/fnx/coupled.py
"""Transient linear thermoelasticity in ONE mixed space (theta, u): theta = T - T0 (temperature change), plane strain, implicit Euler.
heat:  rho_c d(theta)/dt + cpl * T0 beta d(tr eps)/dt - div(k grad theta) = 0      cpl = 1 full coupling, cpl = 0 one-way (weak)
mech:  div sigma = 0,   sigma = 2 mu eps + lambda tr(eps) I - beta theta I          quasi-static
Strip [0, L] x [0, H]: theta = dT at x = 0, insulated elsewhere; u_x = 0 at x = 0, u_y = 0 on y = 0 and y = H (uniaxial strain),
x = L traction-free. The exact 1D solution has the diffusivity kappa = k / rho_c (one-way) or kappa / (1 + delta) (full)."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def moduli(E, nu, alpha):
    mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
    return mu, lmbda, (3 * lmbda + 2 * mu) * alpha

def coupling_delta(E, nu, alpha, rho_c, T0):
    """thermoelastic coupling number  delta = T0 beta^2 / (rho_c (lambda + 2 mu))"""
    mu, lmbda, beta = moduli(E, nu, alpha)
    return T0 * beta**2 / (rho_c * (lmbda + 2 * mu))

def alpha_for_delta(delta, E, nu, rho_c, T0):
    """the expansion coefficient that gives a chosen delta (a 'what if' material)"""
    mu, lmbda, b1 = moduli(E, nu, 1.0)
    return np.sqrt(delta * rho_c * (lmbda + 2 * mu) / T0) / b1

def strip_exact(t, kappa, x=1.0, L=1.0, nterms=400):
    """exact theta(x, t) / dT of  theta_t = kappa theta_xx,  theta(0, t) = dT,  theta_x(L, t) = 0,  theta(x, 0) = 0"""
    n = np.arange(nterms); lam = (2 * n + 1) * np.pi / (2 * L)
    t = np.atleast_1d(np.asarray(t, dtype=float))[:, None]
    return 1 - np.sum(4 / ((2 * n + 1) * np.pi) * np.sin(lam * x) * np.exp(-lam**2 * kappa * t), axis=1)

def t_half(t, y):
    """first time at which y reaches 1/2 (linear interpolation between steps)"""
    i = int(np.argmax(np.asarray(y) >= 0.5))
    return t[i - 1] + (0.5 - y[i - 1]) * (t[i] - t[i - 1]) / (y[i] - y[i - 1])

def run_strip(E=1.0, nu=0.3, alpha=0.1, k=1.0, rho_c=1.0, T0=1.0, dT=1.0, cpl=1.0, L=1.0, H=0.1, nx=64, ny=2,
              dt=2e-3, t_end=1.5, degT=1, degu=2):
    """returns dict: t, TL = mean theta(L, t)/dT, UL = mean u_x(L, t), delta, kappa, beta, reason, and the final w (theta, u)"""
    msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [nx, ny], cell_type=mesh.CellType.triangle)
    el = basix.ufl.mixed_element([basix.ufl.element("Lagrange", "triangle", degT),
                                  basix.ufl.element("Lagrange", "triangle", degu, shape=(2,))])
    W = fem.functionspace(msh, el)                                  # ONE space: sub(0) = theta, sub(1) = u
    mu, lmbda, beta = moduli(E, nu, alpha)
    (th, u), (s, v) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    w_n = fem.Function(W); th_n, u_n = ufl.split(w_n)               # previous step
    eps = lambda w: ufl.sym(ufl.grad(w))
    sig = lambda w, T: 2 * mu * eps(w) + lmbda * ufl.tr(eps(w)) * ufl.Identity(2) - beta * T * ufl.Identity(2)   # plane strain
    idt, c = fem.Constant(msh, 1.0 / dt), fem.Constant(msh, float(cpl))
    a = (rho_c * idt * th * s + c * T0 * beta * idt * ufl.tr(eps(u)) * s + k * ufl.inner(ufl.grad(th), ufl.grad(s))) * ufl.dx \
        + ufl.inner(sig(u, th), eps(v)) * ufl.dx
    Lf = (rho_c * idt * th_n * s + c * T0 * beta * idt * ufl.tr(eps(u_n)) * s) * ufl.dx
    fd = lambda f: mesh.locate_entities_boundary(msh, 1, f)
    left = fd(lambda X: np.isclose(X[0], 0)); tb = fd(lambda X: np.isclose(X[1], 0) | np.isclose(X[1], H))
    bcs = [fem.dirichletbc(fem.Constant(msh, float(dT)), fem.locate_dofs_topological(W.sub(0), 1, left), W.sub(0)),
           fem.dirichletbc(0.0, fem.locate_dofs_topological(W.sub(1).sub(0), 1, left), W.sub(1).sub(0)),
           fem.dirichletbc(0.0, fem.locate_dofs_topological(W.sub(1).sub(1), 1, tb), W.sub(1).sub(1))]
    a, Lf = fem.form(a), fem.form(Lf)
    A = assemble_matrix(a, bcs=bcs); A.assemble()                   # constant in time: factorise once
    ksp = PETSc.KSP().create(comm); ksp.setOperators(A); ksp.setType("preonly")
    ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    right = np.sort(fd(lambda X: np.isclose(X[0], L)))
    dsR = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, right, np.full(len(right), 1, np.int32)))
    thL, uL = fem.form(w_n.sub(0) * dsR(1)), fem.form(w_n.sub(1)[0] * dsR(1))
    w = fem.Function(W); ts, TL, UL = [0.0], [0.0], [0.0]
    for n in range(1, int(round(t_end / dt)) + 1):
        b = assemble_vector(Lf); apply_lifting(b, [a], [bcs])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, bcs)
        ksp.solve(b, w.x.petsc_vec); w.x.scatter_forward()
        w_n.x.array[:] = w.x.array
        ts.append(n * dt)
        TL.append(comm.allreduce(fem.assemble_scalar(thL), op=MPI.SUM) / H / dT)
        UL.append(comm.allreduce(fem.assemble_scalar(uL), op=MPI.SUM) / H)
    return dict(t=np.array(ts), TL=np.array(TL), UL=np.array(UL), delta=T0 * beta**2 / (rho_c * (lmbda + 2 * mu)),
                kappa=k / rho_c, beta=beta, lmbda=lmbda, mu=mu, reason=ksp.getConvergedReason(), w=w, W=W, msh=msh)

## 2 · The thick tube as a cross-section  *(lecture §A1–A4, §B3–B4)*
Quarter of the tube $r_i=1\le r\le r_o=2$, internal pressure $p=1$, $E=1$, $\nu=0.3$; rollers on the two symmetry lines. Three end conditions:
plane strain ($\varepsilon_{zz}=0$), open ends (GPS, $N=0$) and closed ends (GPS, $N$ = the end-cap force on the **quarter**, $p\pi r_i^2/4$).
The first line checks the curved mesh: its area against $3\pi/4$.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from gps import *

ri, ro, p, nu = 1.0, 2.0, 1.0, 0.3
mat = Material(1.0, nu)
A_ = p * ri**2 / (ro**2 - ri**2); B_ = A_ * ro**2; srr, stt = A_ - B_ / ri**2, A_ + B_ / ri**2
lame = {k: ri * (stt - nu * (srr + szz)) for k, szz in (("strain", 2 * nu * A_), ("open", 0.0), ("closed", A_))}

msh, ft = annulus_mesh(ri, ro, 8, 16)
say(f"area {area(msh):.10f}   exact 3 pi / 4 = {3 * np.pi / 4:.10f}   cells {msh.topology.index_map(2).size_global}")
nn = ufl.FacetNormal(msh)
for name, mode, N in (("strain", "strain", 0.0), ("open", "gps", 0.0), ("closed", "gps", p * np.pi * ri**2 / 4)):
    uh, i = solve_2d(msh, ft, mat, 2, mode, N, tractions=[(INNER, -p * nn)], fixed=[(BOTTOM, 1), (LEFT, 0)])
    ur = comm.allreduce(fem.assemble_scalar(fem.form(ufl.dot(uh, -nn) * i["ds"](INNER))), op=MPI.SUM) / (np.pi / 2 * ri)
    say(f"{name:6s}: e0 {i['e0']:+.7f}  int sigma_zz {i['N']:.6f}  mean u_r(r_i) {ur:.7f}  (Lame {lame[name]:.7f})  reason {i['reason']}")

> **✏️ Your turn (5 min).** (i) From Lamé, $\sigma_{rr}+\sigma_{\theta\theta}=2A$ with $A=pr_i^2/(r_o^2-r_i^2)$. Compute $e_0$ by hand for open and closed ends and
> compare. (ii) Why is the axial force of the closed tube $p\pi r_i^2/4$ here? What would $N=p\pi r_i^2$ in this model mean?

> *your answer*

## 3 · Static condensation vs the bordered matrix  *(lecture §A3, §B5)*
`solve_2d` returns the matrix $K$ and the two right-hand sides it solved with. With $\mathbf g=-\mathbf b_1$ (the load of the case $e_0=1$) and
$c=\int(\lambda+2\mu)\,dA$, the extra unknown becomes one more row and column. On a small mesh, **serially**, we can solve that system densely and compare.

In [ ]:
%%fenicsx -np 1
import sys; sys.path.insert(0, "/tmp/fnx")
from gps import *

ri, ro, p, nu = 1.0, 2.0, 1.0, 0.3
mat = Material(1.0, nu)
A_ = p * ri**2 / (ro**2 - ri**2); B_ = A_ * ro**2; srr, stt = A_ - B_ / ri**2, A_ + B_ / ri**2
lame = {k: ri * (stt - nu * (srr + szz)) for k, szz in (("strain", 2 * nu * A_), ("open", 0.0), ("closed", A_))}

msh, ft = annulus_mesh(ri, ro, 4, 8); nn = ufl.FacetNormal(msh); N = p * np.pi * ri**2 / 4
uh, i = solve_2d(msh, ft, mat, 2, "gps", N, tractions=[(INNER, -p * nn)], fixed=[(BOTTOM, 1), (LEFT, 0)])
K = i["A"].convert("dense").getDenseArray()                  # serial only: the whole matrix on this process
f0, g = i["rhs"][0].array.copy(), -i["rhs"][1].array.copy()
c = (mat.lmbda + 2 * mat.mu) * area(msh)
M = np.block([[K, g[:, None]], [g[None, :], np.array([[c]])]])   # [[K, g], [g^T, c]]
x = np.linalg.solve(M, np.r_[f0, N])
say(f"bordered size {M.shape[0]}, max|M - M^T| {np.abs(M - M.T).max():.1e}")
say(f"e0: bordered {x[-1]:.12f}   condensed {i['e0']:.12f}   max|u difference| {np.abs(x[:-1] - uh.x.array).max():.1e}")
say(f"c = (lambda + 2 mu)|Omega_h| = {c:.7f};   N1 = {i['N1']:.7f};   c - g^T K^-1 g = {c - g @ np.linalg.solve(K, g):.7f};   E |Omega_h| = {area(msh):.7f}")

> **✏️ Your turn (5 min).** $N_1$ is the axial force of the case "$e_0=1$, no load". Explain why it equals $E\,|\Omega_h|$ and not $(\lambda+2\mu)|\Omega_h|$.
> Which of the two would you get if the case $e_0=1$ were solved **without** its right-hand side $\mathbf b_1$?

> *your answer*

## 4 · One-way coupling: the heated plate  *(lecture §A5–A6, §C1–C3)*
$[0,1]^2$, steady conduction with $T=0$ on $x=0$ and $T=1$ on $x=1$ (so $T=x$), plane stress, $E=210$ GPa, $\nu=0.3$, $\alpha=10^{-5}$ (units MPa, m).
Two sets of supports: **rollers** on $x=0$ ($u_x=0$) and $y=0$ ($u_y=0$), or a **pin** at $(0,0)$ and a **roller** at $(1,0)$.

> **✏️ Predict first (3 min).** Is $T=x$ harmonic? Which support gives zero stress? What is $\max u_x$ then?

> *your prediction*

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from gps import *

E, nu, alpha = 210e3, 0.3, 1e-5
mat = Material(E, nu, alpha)
msh, ft = rect_mesh(1.0, 1.0, 40, 40)
Th = heat(msh, ft, dirichlet=((LEFT, 0.0), (RIGHT, 1.0)))
X = ufl.SpatialCoordinate(msh)
ufree = alpha * ufl.as_vector([(X[0]**2 - X[1]**2) / 2, X[0] * X[1]])           # free thermal expansion of T = x
pin = lambda P: np.isclose(P[0], 0) & np.isclose(P[1], 0)
roller = lambda P: np.isclose(P[0], 1) & np.isclose(P[1], 0)
for name, kw in (("rollers", dict(fixed=[(LEFT, 0), (BOTTOM, 1)])), ("pin+roller", dict(points=[(pin, 0), (pin, 1), (roller, 1)]))):
    for mode in ("stress", "strain"):
        uh, i = solve_2d(msh, ft, mat, 2, mode, dT=Th, **kw)
        uxm = comm.allreduce(float(uh.x.array[0::2].max()), op=MPI.MAX)
        say(f"{name:10s} plane {mode}: max u_x {uxm:.4e}  max von Mises {dg_max(msh, von_mises(i['sigma'])):.3e}  "
            f"max|sigma_zz| {dg_max(msh, abs(i['sigma'][2, 2])):.3e}  ||u - u_free|| {l2(msh, uh - ufree):.1e}")

In [ ]:
%%fenicsx -np 2
# ✏️ Your turn (10 min): a Gaussian source Q = 1000 exp(-((x-0.5)^2 + (y-0.5)^2) / (2 * 0.1^2)) with T = 0 on all four walls (heat(..., Q=Q, dirichlet=...)).
# Print max T, then max |u| and max von Mises (plane stress) for both sets of supports.
# TODO

> **✏️ Your turn (5 min).** Explain the four lines of the first cell and the Gaussian case in one sentence each. What does an earlier version of this example
> mean when it calls the 14 % difference from $\alpha/2$ "the Poisson effect"?

> *your answer*

## 5 · Full coupling: the strip  *(lecture §A7–A8, §C4–C6)*
`run_strip` solves the $(\theta,\mathbf u)$ system in one mixed space with implicit Euler; `cpl=0` drops $T_0\beta\operatorname{tr}\dot{\boldsymbol\varepsilon}$ (one-way),
`cpl=1` keeps it. A "what if" material with $\delta=0.5$ makes the coupling visible. The cell also stores the exact series for both diffusivities.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from coupled import *

a05 = alpha_for_delta(0.5, 1.0, 0.3, 1.0, 1.0)            # E = 1, nu = 0.3, rho_c = 1, T0 = 1
out = {}
for cpl in (0.0, 1.0):
    r = run_strip(alpha=a05, cpl=cpl, dt=2e-3, t_end=1.5)
    kap = r["kappa"] / (1 + r["delta"] * cpl)
    ex = strip_exact(r["t"], kap)
    out[f"TL{int(cpl)}"], out[f"ex{int(cpl)}"], out["t"] = r["TL"], ex, r["t"]
    say(f"cpl {cpl:.0f}: delta {r['delta']:.4f}  t50 {t_half(r['t'], r['TL']):.5f} (exact {t_half(r['t'], ex):.5f})  "
        f"max|theta(L) - exact| {np.abs(r['TL'] - ex).max():.2e}  u_x(L, 1.5) {r['UL'][-1]:.5f}  reason {r['reason']}")
say(f"ratio t50 full / one-way: {t_half(out['t'], out['TL1']) / t_half(out['t'], out['TL0']):.5f}   (1 + delta = 1.5)")
if comm.rank == 0:
    np.savez("/tmp/fnx/w03_strip.npz", **out)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
D = np.load("/tmp/fnx/w03_strip.npz"); t = D["t"]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t, D["ex0"], "k-", lw=1, label="exact, one-way"); ax.plot(t, D["ex1"], "k--", lw=1, label="exact, full")
ax.plot(t[::25], D["TL0"][::25], "o", label="FEM one-way"); ax.plot(t[::25], D["TL1"][::25], "s", label="FEM full")
ax.axhline(0.5, color="gray", lw=0.6, ls=":")
ax.set(xlabel="t", ylabel="theta(L, t) / dT", title="strip, delta = 0.5"); ax.legend(); ax.grid(alpha=0.3); plt.show()

> **✏️ Your turn (5 min).** Compute $\delta$ for steel ($E=210$ GPa, $\nu=0.3$, $\alpha=1.2\times10^{-5}$ /K, $\rho=7850$ kg/m³, $c=460$ J/(kg K), $T_0=293$ K)
> with `coupling_delta` or by hand. By how much does a one-way model misjudge how fast a steel part heats up? Why does the steady state not tell you?

> *your answer*

## 6 · LLM exercise: thermal stress in a long tube  *(lecture Part D)*
Ask your LLM for a test cell that uses the lab module `gps.py` (paste it, and the API sheet §16) for the thermal stresses of a long thick tube with free
ends: $a=1$, $b=2$, inner wall at $\Delta T=1$, outer at $0$, steady conduction (so $\Delta T=\ln(b/r)/\ln(b/a)$), $E=1$, $\nu=0.3$, $\alpha=1$, no pressure.

```text
Using the attached module gps.py (DOLFINx 0.11), write a %%fenicsx -np 2 cell for a long thick tube, a = 1, b = 2, with free ends (generalized plane
strain, N = 0) and, for comparison, plane strain. Steady temperature: Delta T = 1 on r = a, 0 on r = b (use heat(..., deg=2) on the quarter annulus).
Thermal load only, E = 1, nu = 0.3, alpha = 1, rollers on the symmetry lines. Print the mean hoop stress and the mean axial stress on r = a for
both cases, on three meshes, next to the closed-form solution of Timoshenko and Goodier for a long cylinder with a radial temperature field.
```

```python
# paste here
```

**Check before you trust it**

- [ ] The temperature field is checked first, against $\ln(b/r)/\ln(b/a)$, before any stress is read
- [ ] The hoop stress is $\mathbf t\cdot\boldsymbol\sigma\,\mathbf t$ with the **tangent** $\mathbf t=(-y,x)/r$, not $\sigma_{yy}$
- [ ] Free ends means $\int\sigma_{zz}\,dA=0$ over the **modelled** quarter, not $\sigma_{zz}=0$ pointwise
- [ ] The closed-form solution is for the same end condition as the model; the error falls under refinement
- [ ] `-np 1` and `-np 2` give the same numbers

In [ ]:
%%fenicsx -np 2
# TODO: the LLM's cell (after you checked it)

> *your verdict on the LLM's code*